In [1]:
# ── STEP 1: imports ───────────────────────────────────────
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ── STEP 2: load data ─────────────────────────────────────
df = pd.read_csv("Crop_recommendation.csv")

# ── STEP 3: split X and y ────────────────────────────────
X = df.drop("label", axis=1)
y = df["label"]

# ── STEP 4: train/test split ──────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ── STEP 5: label encode ──────────────────────────────────
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

# ── STEP 6: scale features ────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


In [2]:
dt_model = DecisionTreeClassifier(criterion="gini", random_state=42)
dt_model.fit(X_train, y_train_enc)        # train
dt_pred = dt_model.predict(X_test)        # predict
dt_acc  = accuracy_score(y_test_enc, dt_pred)

print("=" * 40)
print("DECISION TREE")
print("=" * 40)
print(f"Accuracy: {dt_acc * 100:.2f}%")
print(classification_report(y_test_enc, dt_pred,
      target_names=le.classes_))

DECISION TREE
Accuracy: 97.95%
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.80      0.89        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      0.95      0.95        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       0.86      0.90      0.88        20
       maize       0.95      1.00      0.98        20
       mango       1.00      1.00      1.00        20
   mothbeans       0.86      0.95      0.90        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00      1.00       

In [3]:
rf_model = RandomForestClassifier(n_estimators=100,random_state=42)
rf_model.fit(X_train, y_train_enc)        # train — this takes ~5 seconds
rf_pred = rf_model.predict(X_test)        # predict
rf_acc  = accuracy_score(y_test_enc, rf_pred)

print("=" * 40)
print("RANDOM FOREST")
print("=" * 40)
print(f"Accuracy: {rf_acc * 100:.2f}%")
print(classification_report(y_test_enc, rf_pred,
      target_names=le.classes_))

RANDOM FOREST
Accuracy: 99.55%
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.95      0.97        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      1.00      1.00        20
       maize       0.95      1.00      0.98        20
       mango       1.00      1.00      1.00        20
   mothbeans       1.00      1.00      1.00        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00      1.00       

In [4]:
print("=" * 40)
print("COMPARISON")
print("=" * 40)
print(f"Decision Tree : {dt_acc * 100:.2f}%")
print(f"Random Forest : {rf_acc * 100:.2f}%")
print(f"RF improvement: +{(rf_acc - dt_acc) * 100:.2f}%")

COMPARISON
Decision Tree : 97.95%
Random Forest : 99.55%
RF improvement: +1.59%


In [5]:
import joblib

joblib.dump(rf_model, "../crop_app/crop_model.pkl")
joblib.dump(scaler, "../crop_app/scaler.pkl")
joblib.dump(le, "../crop_app/label_encoder.pkl")

print("Saved: crop_model.pkl")
print("Saved: scaler.pkl")
print("Saved: label_encoder.pkl")

loaded_model = joblib.load("../crop_app/crop_model.pkl")
loaded_le    = joblib.load("../crop_app/label_encoder.pkl")

sample = pd.DataFrame([[90, 42, 43, 20.87, 82.0, 6.5, 202.9]],
                       columns=["N","P","K","temperature","humidity","ph","rainfall"])

prediction = loaded_model.predict(sample)
crop_name  = loaded_le.inverse_transform(prediction)[0]
print(f"\nTest prediction: {crop_name.upper()}")

Saved: crop_model.pkl
Saved: scaler.pkl
Saved: label_encoder.pkl

Test prediction: RICE
